In [ ]:
import matplotlib.pyplot as plt

# 设置支持中文的字体（例如 SimHei），同时确保负号能正常显示
# plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['font.sans-serif'] = ['Times New Roman', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import numpy as np

df = pd.read_csv("../data/7.匹配时间/merged_output.csv")

In [ ]:
df

# 计算发生时间到判决时间的天数

In [ ]:
# 1) 确保两列都是 datetime
df["裁判日期"] = pd.to_datetime(df["裁判日期"], errors="coerce")
df["case_dt_hour"] = pd.to_datetime(df["case_dt_hour"], errors="coerce")

# 2) 计算相差天数（case_dt_hour - 裁判日期）
#    - 用 .dt.days 得到“整数天”（向下取整到天）
df["judge_time"] = (df["裁判日期"] - df["case_dt_hour"]).dt.days

print("清理前剩余行数：", len(df))

# 去除掉负数天数

In [ ]:
# 3) 统计 judge_time 为负数的个数（会自动忽略 NaN）
neg_mask = df["judge_time"] < 0
neg_count = neg_mask.sum()
print("judge_time < 0 的行数：", neg_count)

# 3) 去掉 judge_time 为负数的行（保留 judge_time >= 0 或 NaN）
df = df.loc[~neg_mask].copy()

print("清理前剩余行数：", len(df))


In [ ]:
df

# 空间连接到天地图城市

In [ ]:
# 2) 读城市边界 GeoJSON
city_gdf = gpd.read_file(r"E:\202512LLM交通\figure\china_geodata/中国_市_eng.shp")

In [ ]:
city_gdf

In [ ]:
# 转换经纬度

# 3) 只保留需要字段，修一下几何（可选但推荐）
city_gdf = city_gdf[["name_eng", "geometry"]].copy()
city_gdf["geometry"] = city_gdf["geometry"].buffer(0)  # 修复少量无效几何（常见且有效）

# 4) CRS 对齐（你的 geojson 通常是经纬度 WGS84，但有时 crs 为空）
#    点数据一定是 EPSG:4326（经纬度）
if city_gdf.crs is None:
    city_gdf = city_gdf.set_crs(epsg=4326)
else:
    city_gdf = city_gdf.to_crs(epsg=4326)

# 5) 把 lng/lat 转成数值，生成点 GeoDataFrame
df["lng"] = pd.to_numeric(df["lng"], errors="coerce")
df["lat"] = pd.to_numeric(df["lat"], errors="coerce")

points_gdf = gpd.GeoDataFrame(
    df[["lng", "lat"]].copy(),
    geometry=gpd.points_from_xy(df["lng"], df["lat"]),
    crs="EPSG:4326")

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# =========================
# 参数区：按需修改
# =========================
LNG_COL = "lng"
LAT_COL = "lat"
CITY_NAME_COL = "name_eng"          # city_gdf 里城市名称字段
OUTPUT_COL = "city_name"        # 写回 df 的列名

PREDICATE = "within"            # "within" 或 "intersects"
USE_NEAREST_FALLBACK = True     # 是否对 NaN 做 nearest 兜底
MAX_NEAREST_DIST_M = None       # e.g. 50000 表示 50km，None 表示不限制

# =========================
# 0) 基础清洗：经纬度有效性
# =========================
df = df.copy()

# 转数值，非法变 NaN
df[LNG_COL] = pd.to_numeric(df[LNG_COL], errors="coerce")
df[LAT_COL] = pd.to_numeric(df[LAT_COL], errors="coerce")

# 可选：过滤明显不合法的经纬度
valid_coord = (
    df[LNG_COL].between(-180, 180, inclusive="both") &
    df[LAT_COL].between(-90, 90, inclusive="both")
)

# =========================
# 1) 构建点 GeoDataFrame（WGS84）
# =========================
points_gdf = gpd.GeoDataFrame(
    df.loc[valid_coord].copy(),
    geometry=gpd.points_from_xy(df.loc[valid_coord, LNG_COL], df.loc[valid_coord, LAT_COL]),
    crs="EPSG:4326"
)

# =========================
# 2) 确保 city_gdf 有 CRS，并与点一致（用于 within/intersects）
# =========================
if city_gdf.crs is None:
    raise ValueError("city_gdf.crs is None：请先给城市面数据设置正确的 CRS（例如 EPSG:4326）。")

# 先把城市面转换到 EPSG:4326，以便和 points_gdf 做空间连接
city_4326 = city_gdf.to_crs(points_gdf.crs)

# =========================
# 3) 空间连接：点落在哪个城市面
# =========================
joined = gpd.sjoin(
    points_gdf,
    city_4326[[CITY_NAME_COL, "geometry"]],
    how="left",
    predicate=PREDICATE
)

# 处理：极少数一个点命中多个面 -> 取第一个
city_name = joined[CITY_NAME_COL].groupby(joined.index).first()

# 先创建输出列
df[OUTPUT_COL] = pd.NA
# 回填到原 df（对齐索引）
df.loc[city_name.index, OUTPUT_COL] = city_name

# =========================
# 4) nearest 兜底：只对仍是 NaN 的点做
# =========================
if USE_NEAREST_FALLBACK:
    mask_na = df[OUTPUT_COL].isna() & valid_coord

    if mask_na.any():
        # 只取需要兜底的点
        points_na = gpd.GeoDataFrame(
            df.loc[mask_na].copy(),
            geometry=gpd.points_from_xy(df.loc[mask_na, LNG_COL], df.loc[mask_na, LAT_COL]),
            crs="EPSG:4326"
        )

        # --- 关键修复：投影到米制 CRS 再 nearest（避免 warning & 距离更准确） ---
        # 自动估算一个适合此区域的 UTM CRS（单位：米）
        proj_crs = city_4326.estimate_utm_crs()

        points_proj = points_na.to_crs(proj_crs)
        city_proj = city_4326.to_crs(proj_crs)

        nearest = gpd.sjoin_nearest(
            points_proj,
            city_proj[[CITY_NAME_COL, "geometry"]],
            how="left",
            distance_col="nearest_dist_m"
        )

        # 如果你想限制最大兜底距离（比如海上点别硬匹配）
        if MAX_NEAREST_DIST_M is not None:
            nearest.loc[nearest["nearest_dist_m"] > MAX_NEAREST_DIST_M, CITY_NAME_COL] = pd.NA

        # --- 关键修复：按 index 对齐回填，避免 .values 顺序错位 ---
        df.loc[nearest.index, OUTPUT_COL] = nearest[CITY_NAME_COL]

In [ ]:
df

In [ ]:
# 筛选列
cols = ["death", "judge_time", "city_name"]
df = df[cols]

In [ ]:
df

# 统计缺失值

In [ ]:
# 统计 judge_time 和 death 是否有缺失值，以及缺失多少
cols = ["death", "judge_time"]

missing_cnt = df[cols].isna().sum()               # 每列缺失个数
missing_ratio = df[cols].isna().mean()            # 每列缺失比例(0~1)

missing_summary = pd.DataFrame({
    "missing_count": missing_cnt,
    "missing_ratio": missing_ratio
}).reset_index().rename(columns={"index": "column"})

print(missing_summary)

# 如果你还想看“任意一个字段缺失”的行数：
rows_with_missing = df[cols].isna().any(axis=1).sum()
print("rows_with_missing(death or judge_time):", rows_with_missing)


# 清洗数据

In [ ]:
df = df.copy()

df.loc[:, "death"] = pd.to_numeric(df["death"], errors="coerce")
df.loc[:, "judge_time"] = pd.to_numeric(df["judge_time"], errors="coerce")

In [ ]:
df

# judge_time：算平均值（mean）
# death_1_ratio：算该城市 death==1.0 的占比（= 1.0 的个数 / 该城市总数）
# 保留每个城市的样本量 n

In [ ]:
import numpy as np
import pandas as pd

# df: 包含 death, judge_time, city_name 三列

result = (df.groupby("city_name", as_index=False).agg(judge_time_mean=("judge_time", "mean"),n=("death", "size"),death_1_cnt=("death", lambda s: (s == 1.0).sum()),))

result["judge_time_mean"] = result["judge_time_mean"].round().astype("Int64")  # 可处理缺失值
result["death_1_ratio"] = result["death_1_cnt"] / result["n"]

# 如果你不想要 cnt，只想要占比，可以最后 drop 掉
result = result.drop(columns=["death_1_cnt"])

# result.to_csv('./TEST.csv')
result

# 筛选样本量 保留前30？或者不筛选

In [ ]:
# 把 result 本身也直接变成前n条
result_top = result.sort_values(by='n', ascending=False).head(366).reset_index(drop=True)

result_top

# 绘制散点图

## 案件数量 vs 判决天数

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

df = result_top.copy()
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["n", "judge_time_mean", "city_name"])
df = df[df["n"] > 0]

x = df["n"].astype(float).to_numpy()
y = df["judge_time_mean"].astype(float).to_numpy()
labels = df["city_name"].astype(str).to_numpy()

# 线性回归：y = b0 + b1*x
X = sm.add_constant(x)
model = sm.OLS(y, X).fit()

# 拟合线 + 置信区间
x_grid = np.linspace(x.min(), x.max(), 300)
X_grid = sm.add_constant(x_grid)

pred = model.get_prediction(X_grid).summary_frame(alpha=0.05)  # 95% CI
y_hat = pred["mean"].to_numpy()
ci_low = pred["mean_ci_lower"].to_numpy()
ci_high = pred["mean_ci_upper"].to_numpy()

# ===== 颜色控制区域 =====
point_color = "#d1e5f5"
line_color  = "#d62728"
band_color  = "#d62728"
band_alpha  = 0.18
edge_color  = "#33729d"
# =======================

plt.figure(figsize=(5, 5),dpi=300)

# 散点
plt.scatter(
    x, y,
    s=13,
    c=point_color,
    edgecolors=edge_color,
    linewidths=0.5,
    alpha=1
)

# 拟合线
plt.plot(
    x_grid, y_hat,
    color=line_color,
    linewidth=1,
    linestyle="-"
)

# 区间带
plt.fill_between(
    x_grid, ci_low, ci_high,
    color=band_color,
    alpha=band_alpha,
    linewidth=0)

# ====== 标注 city_name ======
# 如果你担心 366 个城市都标注会太挤，可以把 annotate_all=False 并设置 top_n
annotate_all = False
top_n = 30  # annotate_all=False 时，标注 n 最大的前 top_n 个点（你也可以换成别的规则）

if annotate_all:
    idx_to_annotate = range(len(df))
else:
    idx_to_annotate = np.argsort(x)[-top_n:]  # 例：只标注 n 最大的 top_n

for i in idx_to_annotate:
    plt.annotate(
        labels[i],
        (x[i], y[i]),
        textcoords="offset points",
        xytext=(0, 3),           # 向上偏移 3 个点
        ha="center",
        va="bottom",
        fontsize=6,
        color="#2b2b2b",
        alpha=0.9,
        clip_on=True)
# ============================

plt.xlabel("Number of Included Court-Adjudicated Cases")
plt.ylabel("Mean Accident-to-Judgment Interval (days)")
plt.grid(True, alpha=0.3)

plt.savefig(
    # '../figure/fig5/case_num_judge_time.png',
    '../figure/fig5/case_num_judge_time_NOLABEL.png',
    dpi=300,
    bbox_inches='tight',
    pad_inches=0.1
)
plt.show()


## 案件数量 vs 死亡率

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

df = result_top.copy()
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["n", "death_1_ratio", "city_name"])
df = df[df["n"] > 0]

x = df["n"].astype(float).to_numpy()
y = df["death_1_ratio"].astype(float).to_numpy()
labels = df["city_name"].astype(str).to_numpy()

# 线性回归：y = b0 + b1*x
X = sm.add_constant(x)
model = sm.OLS(y, X).fit()

# 拟合线 + 置信区间
x_grid = np.linspace(x.min(), x.max(), 300)
X_grid = sm.add_constant(x_grid)

pred = model.get_prediction(X_grid).summary_frame(alpha=0.05)  # 95% CI
y_hat = pred["mean"].to_numpy()
ci_low = pred["mean_ci_lower"].to_numpy()
ci_high = pred["mean_ci_upper"].to_numpy()

# ===== 颜色控制区域 =====
point_color = "#d1e5f5"
line_color  = "#d62728"
band_color  = "#d62728"
band_alpha  = 0.18
edge_color  = "#33729d"
# =======================

plt.figure(figsize=(5, 5),dpi=300)

# 散点
plt.scatter(
    x, y,
    s=13,
    c=point_color,
    edgecolors=edge_color,
    linewidths=0.5,
    alpha=1
)

# 拟合线
plt.plot(
    x_grid, y_hat,
    color=line_color,
    linewidth=1,
    linestyle="-"
)

# 区间带
plt.fill_between(
    x_grid, ci_low, ci_high,
    color=band_color,
    alpha=band_alpha,
    linewidth=0)

# ====== 标注 city_name ======
# 如果你担心 366 个城市都标注会太挤，可以把 annotate_all=False 并设置 top_n
# annotate_all = False
# top_n = 30  # annotate_all=False 时，标注 n 最大的前 top_n 个点（你也可以换成别的规则）

# if annotate_all:
#     idx_to_annotate = range(len(df))
# else:
#     idx_to_annotate = np.argsort(x)[-top_n:]  # 例：只标注 n 最大的 top_n

# for i in idx_to_annotate:
#     plt.annotate(
#         labels[i],
#         (x[i], y[i]),
#         textcoords="offset points",
#         xytext=(0, 3),           # 向上偏移 3 个点
#         ha="center",
#         va="bottom",
#         fontsize=6,
#         color="#2b2b2b",
#         alpha=0.9,
#         clip_on=True)
# ============================

plt.xlabel("Number of Included Court-Adjudicated Cases")
plt.ylabel("Proportion of Included Cases Involving a Fatality")
plt.grid(True, alpha=0.3)
plt.ylim(0, 1) # 限制纵坐标轴的范围

plt.savefig(
    # '../figure/fig5/case_num_death_ratio.png',
    '../figure/fig5/case_num_death_ratio_NOLABEL.png',
    dpi=300,
    bbox_inches='tight',
    pad_inches=0.1
)
plt.show()
